# DeepForest model training

`src/` recovers bird positions from screenshots where a point-counting tool baked
coloured dots into the pixels. `scripts/export_dataset.py` maps those positions onto the
original photographs and writes a DeepForest CSV. This notebook answers the question the
CSV cannot: **can a detector learn from data recovered that way?**

Score the pretrained DeepForest bird model on photographs it will never train on. Train
it on the recovered annotations. Score it again on the same photographs. The difference
is what the recovered data added.

```
413 frames exported by the pipeline
  349  train
   60  held out, never seen during training
```

## What it came back with

```
                  pretrained    fine-tuned
mAP@50              0.036         0.087       2.4x
best F1             0.288         0.333       both peak at threshold 0.20
per frame           40 of 60 held-out frames improved, 15 worse, 5 level
```

Precision and recall rise together at 0.05, 0.10 and 0.20. Above 0.30 the fine-tuned
model's recall falls, 0.207 to 0.122, so run it near 0.10 to 0.20 rather than at the
default. Everything here is scored against the recovered annotations, which measures
agreement with the pipeline on unseen photographs. `docs/training_analysis.md` carries the
full reading, including what it does not say.

## Running it

Kaggle or Colab, one T4, about 4 hours. **Accelerator: GPU. Internet: on**, or neither
pip nor the photograph download works. On Kaggle, **Save Version, Save and Run All** runs
the whole thing in the background without the browser open.

## 1. Where this runs, and where things go

The annotations are cloned with the repository rather than uploaded, because the export
already ran. It ran on a Colab CPU runtime in six chunks of about 21 minutes, roughly two
hours in total, because it needs both the screenshot and its paired original for every
frame and a Colab session is shorter than that:

```
python scripts/build_manifest.py                  # pair every screenshot with its original
python scripts/build_benchmark.py --per-cell 57   # 1,197 candidates, stratified by
                                                  # survey year and density band
python scripts/export_dataset.py --out results/dataset_scaled
```

```
1,076 pairs -> 458 pass selection (43%) -> 413 exported -> 118,270 boxes
```

`export_dataset.py` runs the whole recovery pipeline per frame: locate the dialog, parse
its legend, detect the dots, assign each to a legend row, map screenshot pixels to
original pixels, and measure a box from that frame's own birds. Everything from here on
reads its output.

`results/dataset_scaled/annotations_deepforest.csv` holds the six columns DeepForest
reads. `exported_frames.csv` beside it records what was measured on each frame. Only the
first is needed below.

In [ ]:
import os, sys, glob, json, re, time, subprocess, shutil
import pandas as pd

# Kaggle keeps /kaggle/working after the session and discards /kaggle/tmp. Anywhere
# else, both land beside the notebook.
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("e4_out")
TMP = "/kaggle/tmp" if ON_KAGGLE else os.path.abspath("e4_tmp")
os.makedirs(WORK, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Kaggle: Settings -> Accelerator -> GPU T4. "
                     "Colab: Runtime -> Change runtime type -> T4 GPU.")
print("gpu:", torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("disk free:", round(shutil.disk_usage(TMP).free / 1e9), "GB")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepforest==2.1.0"],
               check=True)
import deepforest
print("deepforest", deepforest.__version__)

# The dataset lives in the repository, so nothing has to be uploaded by hand.
REPO = f"{TMP}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/vickysharma-prog/Recovering-computer-vision-annotations.git",
        REPO], check=True)
sys.path.insert(0, REPO)

ann = pd.read_csv(f"{REPO}/results/dataset_scaled/annotations_deepforest.csv")
ann = ann.drop_duplicates()
print(f"\n{len(ann):,} boxes over {ann.image_path.nunique()} frames")

## 2. The photographs

Public bucket, no credentials. The CSV names each photograph by the path it has in the
bucket, so the download follows the CSV rather than a hand-written list. About 20 minutes.

In [ ]:
import urllib.parse, urllib.request

BUCKET = "https://twi-aviandata.s3.amazonaws.com/avian_monitoring/"
ORIGS = f"{TMP}/photos"
need = sorted(ann.image_path.unique())

t0, missing = time.time(), []
for i, rel in enumerate(need, 1):
    out = os.path.join(ORIGS, rel)
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        try:
            urllib.request.urlretrieve(BUCKET + urllib.parse.quote(rel), out)
        except Exception:
            missing.append(rel)
    if i % 50 == 0 or i == len(need):
        print(f"  {i}/{len(need)}  {time.time() - t0:.0f}s", flush=True)

print(f"\n{len(need) - len(missing)}/{len(need)} photographs, {time.time() - t0:.0f}s")
if missing:
    print(f"{len(missing)} could not be fetched; those frames drop out")

## 3. Box sizes are per frame, and that is deliberate

Each frame's box was measured from that frame's own birds (`src/birdsize.py`). Eleven
years of surveys flew focal lengths from 28mm to 300mm, so a bird spans 10px in one frame
and 70px in another. One fixed box would be wrong nearly everywhere.

In [ ]:
side = (ann.xmax - ann.xmin).round()
by_frame = ann.assign(side=side).groupby("image_path").side.median()

print(f"box side, per box  : {side.min():.0f}-{side.max():.0f}px, "
      f"median {side.median():.0f}px")
print(f"box side, per frame: {by_frame.min():.0f}-{by_frame.max():.0f}px, "
      f"median {by_frame.median():.0f}px")
print(f"\n{len(by_frame)} frames; a single fixed size would be wrong on most of them")

## 4. Tiles

DeepForest trains on crops, not on 15-megapixel photographs. `split_raster` cuts each one
into overlapping tiles and rewrites the boxes into tile coordinates.

The assertion matters. A training set that quietly lost a third of its annotations still
trains and still reports a number. The ratio comes out above 1.0 rather than at 1.0
because a 0.15 overlap writes a box straddling a tile edge into both tiles.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from deepforest.preprocess import split_raster

COLS = ["image_path", "xmin", "ymin", "xmax", "ymax", "label"]
PATCH, OVERLAP = 400, 0.15
TILES = f"{TMP}/tiles"

# One class. The full CSV keeps all the species names and they stay recoverable through
# frame and legend_row; DeepForest's bird model is single-class.
ann["label"] = "Bird"

t0, tiles, failed = time.time(), [], []
paths = sorted(ann.image_path.unique())
for i, rel in enumerate(paths, 1):
    src = os.path.join(ORIGS, rel)
    if not os.path.exists(src):
        continue
    g = ann[ann.image_path == rel][COLS].copy()
    g["image_path"] = os.path.basename(rel)
    try:
        out = split_raster(annotations_file=g, path_to_raster=src,
                           root_dir=os.path.dirname(src), save_dir=TILES,
                           patch_size=PATCH, patch_overlap=OVERLAP)
    except Exception as exc:
        failed.append((os.path.basename(rel), type(exc).__name__))
        continue
    out["frame"] = os.path.splitext(os.path.basename(rel))[0]
    tiles.append(out)
    if i % 50 == 0 or i == len(paths):
        print(f"  tiled {i}/{len(paths)}  {time.time() - t0:.0f}s", flush=True)

tiled = pd.concat(tiles, ignore_index=True)
covered = ann[ann.image_path.map(lambda p: os.path.exists(os.path.join(ORIGS, p)))]
ratio = len(tiled) / len(covered)
print(f"\n{len(covered):,} boxes -> {len(tiled):,} on {tiled.image_path.nunique():,} "
      f"tiles ({ratio:.2f} per original; overlap duplicates boundary boxes)")
if failed:
    print(f"{len(failed)} frames failed to tile:", failed[:5])
assert ratio > 0.9, f"tiling lost boxes: only {ratio:.2f} survived"

## 5. Split by frame, never by dot

Two dots from the same photograph on either side of the split is not a held-out test: the
model has already seen that background, that colony, that light. Whole frames go one way
or the other, and the assertion says so.

In [ ]:
frames = sorted(tiled.frame.unique())
rng = np.random.default_rng(0)
rng.shuffle(frames)
test_f, train_f = set(frames[:60]), set(frames[60:])

train = tiled[tiled.frame.isin(train_f)][COLS]
test = tiled[tiled.frame.isin(test_f)][COLS]
train.to_csv(f"{WORK}/train.csv", index=False)
test.to_csv(f"{WORK}/test_pipeline.csv", index=False)

assert not (train_f & test_f), "a frame is on both sides of the split"
print(f"train  {len(train):,} boxes on {len(train_f)} frames")
print(f"test   {len(test):,} boxes on {len(test_f)} frames, "
      f"{test.image_path.nunique():,} tiles")

## 6. Autocast off, and checked

SAM 3 leaves a global bfloat16 autocast enabled that survives deleting the model.
DeepForest then trains without error and returns every score as 1.0. Nothing here runs
SAM 3, but a silent 1.0 is worth one line to rule out.

In [ ]:
torch.set_autocast_enabled(False)
assert not torch.is_autocast_enabled(), "autocast is on; every score would come back 1.0"
print("autocast off")

## 7. Before, train, after

The pretrained model is scored first, so the comparison has both halves. Same test set
both times; only the model changes.

**mAP** is read at `score_thresh 0.01` so nothing is cut before the curve is integrated.
**Precision and recall** are read at four operating points, because a single threshold can
flatter whichever model happens to suit it.

Three settings here each cost hours to find:

- `config.score_thresh` never reaches the network. `model.model.score_thresh` does.
- `evaluate()` is deprecated in DeepForest 2.0 and reports no mAP. Use
  `trainer.validate()`.
- `trainer.validate()` returns only losses unless
  `config.validation.val_accuracy_interval` is 1. It defaults to 20 and a standalone
  validate runs at epoch 0, so the full pass runs and the metrics are skipped.

Three epochs, not fifteen. One epoch here is about 140,000 box presentations, so three is
already twice the exposure an 18-frame run got at fifteen. Batch size, learning rate,
patch size and overlap are held at that run's values, so the only thing that changes
between the two experiments is how much data there is.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
from deepforest import main as df_main

CKPT = f"{WORK}/E4"
os.makedirs(CKPT, exist_ok=True)

THRESHOLDS = [0.05, 0.10, 0.20, 0.30]
KEEP = ["map", "map_50", "map_75", "box_precision", "box_recall"]


def score(m, thresh, tag):
    m.model.score_thresh = thresh
    m.config.validation.csv_file = f"{WORK}/test_pipeline.csv"
    m.config.validation.root_dir = TILES
    m.config.validation.val_accuracy_interval = 1
    m.create_trainer()
    out = m.trainer.validate(m)[0]
    got = {k: float(out[k]) for k in KEEP if k in out}
    if not got:
        raise RuntimeError(f"no metrics came back, only: {sorted(out)}")
    print(f"  {tag:8s} {thresh:.2f}   " +
          "  ".join(f"{k} {v:.3f}" for k, v in got.items()), flush=True)
    return got


def run_all(m, who):
    res = {"map": score(m, 0.01, who)}
    for th in THRESHOLDS:
        res[f"t{th}"] = score(m, th, who)
    return res


print("BEFORE: pretrained DeepForest, untouched\n")
base = df_main.deepforest()
base.load_model("weecology/deepforest-bird")
before = run_all(base, "before")
del base
torch.cuda.empty_cache()

print("\nTRAINING on the recovered annotations\n")
model = df_main.deepforest()
model.load_model("weecology/deepforest-bird")
model.config.label_dict = {"Bird": 0}
model.config.num_classes = 1
model.config.train.csv_file = f"{WORK}/train.csv"
model.config.train.root_dir = TILES
model.config.train.epochs = 3
model.config.train.lr = 0.0001
model.config.batch_size = 4
model.config.validation.csv_file = None

model.create_trainer(callbacks=[ModelCheckpoint(
    dirpath=CKPT, filename="e4-{epoch}", save_top_k=-1, every_n_epochs=1)])
t0 = time.time()
model.trainer.fit(model)
print(f"\ntrained in {time.time() - t0:.0f}s")

print("\nAFTER: same model, trained on our data\n")
after = run_all(model, "after")

rows = [dict(metric="mAP", before=before["map"].get("map"),
             after=after["map"].get("map")),
        dict(metric="mAP@50", before=before["map"].get("map_50"),
             after=after["map"].get("map_50"))]
for th in THRESHOLDS:
    b, a = before[f"t{th}"], after[f"t{th}"]
    rows += [dict(metric=f"precision @ {th}", before=b.get("box_precision"),
                  after=a.get("box_precision")),
             dict(metric=f"recall @ {th}", before=b.get("box_recall"),
                  after=a.get("box_recall"))]
res = pd.DataFrame(rows)
res["change"] = (res.after - res.before).round(4)

print("\n" + "=" * 62)
print("BEFORE vs AFTER, 60 held-out frames, same test set both times")
print("=" * 62)
print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

model.trainer.save_checkpoint(f"{CKPT}/e4_final.ckpt")
res.to_csv(f"{WORK}/e4_results.csv", index=False)
json.dump(dict(train_boxes=len(train), test_boxes=len(test),
               train_frames=len(train_f), test_frames=len(test_f),
               epochs=3, lr=0.0001, batch=4, patch=PATCH, overlap=OVERLAP,
               before=before, after=after),
          open(f"{WORK}/e4_metrics.json", "w"), indent=2)
print(f"\nsaved to {WORK}: e4_results.csv, e4_metrics.json, E4/")

## 8. What it looks like on a photograph

Two jobs in one cell.

**A control the table above lacks.** `config.batch_size = 4` was set for training and
carries into every validation pass after it, so the pretrained model was scored at batch
size 1 and the fine-tuned one at 4. mAP accumulates over the epoch and cannot be affected,
but precision and recall were not controlled for it, and best F1 is built from those two.
Here both models go through the same forward pass, one tile at a time, over all held-out
tiles.

**The figures.** The tile scan says which frames gained most, and the five largest are
drawn whole, pretrained beside fine-tuned, with a close-up on the densest part of each.
Green is the recovered annotation, red is a prediction sitting on one, orange dashed is a
prediction with nothing labelled under it. Orange is not necessarily wrong: it is either a
false positive or a bird the pipeline missed.

Note that `predict_image` leaves the image on the CPU while the weights sit on the GPU, so
the tensor is placed by hand below.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from matplotlib.lines import Line2D
from PIL import Image
from torchvision.ops import nms

Image.MAX_IMAGE_PIXELS = None
FIGS = f"{WORK}/figures"
os.makedirs(FIGS, exist_ok=True)

SHOW_THRESH, IOU, N_FRAMES, ZOOM = 0.10, 0.40, 5, 700
NMS_IOU = 0.15

test = pd.read_csv(f"{WORK}/test_pipeline.csv")
tile_frame = {t: re.sub(r"_\d+\.(png|tif|jpg|jpeg)$", "", t, flags=re.I)
              for t in test.image_path.unique()}

base = df_main.deepforest()
base.load_model("weecology/deepforest-bird")
base.model.score_thresh = SHOW_THRESH
base.to("cuda"); base.model.eval()
model.model.score_thresh = SHOW_THRESH
model.to("cuda"); model.model.eval()
DEV = next(model.model.parameters()).device


@torch.no_grad()
def raw(m, arr):
    t = torch.from_numpy(arr.astype("float32") / 255.0).permute(2, 0, 1).to(DEV)
    out = m.model([t])[0]
    keep = out["scores"] >= SHOW_THRESH
    return (out["boxes"][keep].cpu().numpy().astype(float),
            out["scores"][keep].cpu().numpy().astype(float))


def load(p):
    return np.array(Image.open(p).convert("RGB"))


@torch.no_grad()
def whole_frame(m, img):
    """Sliding window, then non-maximum suppression to merge the windows."""
    h, w = img.shape[:2]
    step = int(PATCH * (1 - OVERLAP))
    xs = list(range(0, max(w - PATCH, 0) + 1, step)) or [0]
    ys = list(range(0, max(h - PATCH, 0) + 1, step)) or [0]
    if xs[-1] + PATCH < w:
        xs.append(w - PATCH)
    if ys[-1] + PATCH < h:
        ys.append(h - PATCH)
    boxes, scores = [], []
    for y in ys:
        for x in xs:
            b, s = raw(m, img[y:y + PATCH, x:x + PATCH])
            if len(b):
                boxes.append(b + np.array([x, y, x, y]))
                scores.append(s)
    if not boxes:
        return np.empty((0, 4)), np.empty(0)
    boxes, scores = np.vstack(boxes), np.concatenate(scores)
    k = nms(torch.tensor(boxes, dtype=torch.float32),
            torch.tensor(scores, dtype=torch.float32), NMS_IOU).numpy()
    return boxes[k], scores[k]


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ax0, ay0, ax1, ay1 = (a[:, i][:, None] for i in range(4))
    bx0, by0, bx1, by1 = (b[:, i][None, :] for i in range(4))
    iw = np.clip(np.minimum(ax1, bx1) - np.maximum(ax0, bx0), 0, None)
    ih = np.clip(np.minimum(ay1, by1) - np.maximum(ay0, by0), 0, None)
    inter = iw * ih
    union = (ax1 - ax0) * (ay1 - ay0) + (bx1 - bx0) * (by1 - by0) - inter
    return np.where(union > 0, inter / union, 0.0)


def match(truth, box, score):
    """Greedy by confidence, one prediction per labelled bird."""
    hit = np.zeros(len(box), bool)
    if len(truth) == 0 or len(box) == 0:
        return hit
    M = iou_matrix(box, truth)
    taken = np.zeros(len(truth), bool)
    for i in np.argsort(-score):
        free = np.where(taken, -1.0, M[i])
        j = int(np.argmax(free))
        if free[j] >= IOU:
            taken[j] = True
            hit[i] = True
    return hit


rows, t0 = [], time.time()
names = sorted(tile_frame)
for k, name in enumerate(names, 1):
    img = load(os.path.join(TILES, name))
    truth = test[test.image_path == name][["xmin", "ymin", "xmax", "ymax"]].to_numpy(float)
    pb, sb = raw(base, img)
    pa, sa = raw(model, img)
    hb, ha = match(truth, pb, sb), match(truth, pa, sa)
    rows.append(dict(tile=name, frame=tile_frame[name], labelled=len(truth),
                     pre_found=len(pb), pre_hit=int(hb.sum()),
                     ft_found=len(pa), ft_hit=int(ha.sum()),
                     gain=int(ha.sum()) - int(hb.sum())))
    if k % 250 == 0 or k == len(names):
        print(f"  scanned {k}/{len(names)}  {time.time() - t0:.0f}s", flush=True)

scan = pd.DataFrame(rows)
scan.to_csv(f"{WORK}/e4_tile_scan.csv", index=False)
tot = scan[["labelled", "pre_found", "pre_hit", "ft_found", "ft_hit"]].sum()
print(f"\nall {len(scan):,} held-out tiles, both models one tile at a time")
for tag, found, hit in (("pretrained", tot.pre_found, tot.pre_hit),
                        ("fine-tuned", tot.ft_found, tot.ft_hit)):
    p, r = hit / max(found, 1), hit / max(tot.labelled, 1)
    f1 = 0.0 if p + r == 0 else 2 * p * r / (p + r)
    print(f"  {tag:11s} drew {found:6d}   on a labelled bird {hit:6d}   "
          f"P {p:.3f}   R {r:.3f}   F1 {f1:.3f}")

byframe = scan.groupby("frame").agg(
    labelled=("labelled", "sum"), pre_hit=("pre_hit", "sum"),
    ft_hit=("ft_hit", "sum"), gain=("gain", "sum")).reset_index()
byframe = byframe.sort_values("gain", ascending=False)
byframe.to_csv(f"{WORK}/e4_frame_scan.csv", index=False)
up = int((byframe.gain > 0).sum())
down = int((byframe.gain < 0).sum())
print(f"  per frame: fine-tuned found more on {up}, fewer on {down}, "
      f"level on {len(byframe) - up - down}")

rel_of = {os.path.splitext(os.path.basename(p))[0]: p for p in ann.image_path.unique()}
LEG = [Line2D([], [], color="#00d18b", lw=1.8, label="recovered annotation"),
       Line2D([], [], color="#ff3b30", lw=1.8, label="model, on a labelled bird"),
       Line2D([], [], color="#ffb020", lw=1.8, ls="--",
              label="model, nothing labelled there")]


def draw(ax, img, truth, box, hit, title, lw, crop=None):
    ax.imshow(img)
    for x0, y0, x1, y1 in truth:
        ax.add_patch(mp.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                  edgecolor="#00d18b", lw=lw))
    for (x0, y0, x1, y1), h in zip(box, hit):
        ax.add_patch(mp.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                  edgecolor="#ff3b30" if h else "#ffb020",
                                  lw=lw, ls="-" if h else "--"))
    if crop is not None:
        cx0, cy0, cx1, cy1 = crop
        ax.add_patch(mp.Rectangle((cx0, cy0), cx1 - cx0, cy1 - cy0, fill=False,
                                  edgecolor="white", lw=1.8, ls=":"))
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def busiest_window(truth, w, h, side):
    """Where the close-up sits: the densest patch of labelled birds."""
    if len(truth) == 0:
        return (0, 0, min(side, w), min(side, h))
    cx = (truth[:, 0] + truth[:, 2]) / 2
    cy = (truth[:, 1] + truth[:, 3]) / 2
    best, at = -1, (0, 0)
    for gx in np.linspace(0, max(w - side, 0), 16):
        for gy in np.linspace(0, max(h - side, 0), 16):
            n = int(((cx >= gx) & (cx < gx + side) & (cy >= gy) & (cy < gy + side)).sum())
            if n > best:
                best, at = n, (gx, gy)
    x0, y0 = at
    return (x0, y0, min(x0 + side, w), min(y0 + side, h))


for r, (_, row) in enumerate(byframe.head(N_FRAMES).iterrows(), 1):
    frame = row.frame
    rel = rel_of.get(frame)
    if rel is None:
        continue
    img = load(os.path.join(ORIGS, rel))
    h, w = img.shape[:2]
    truth = ann[ann.image_path == rel][["xmin", "ymin", "xmax", "ymax"]].to_numpy(float)
    pb, sb = whole_frame(base, img)
    pa, sa = whole_frame(model, img)
    hb, ha = match(truth, pb, sb), match(truth, pa, sa)

    cx0, cy0, cx1, cy1 = busiest_window(truth, w, h, ZOOM)
    sub = img[int(cy0):int(cy1), int(cx0):int(cx1)]
    off = np.array([cx0, cy0, cx0, cy0])

    def inside(b):
        if len(b) == 0:
            return np.zeros(0, bool)
        mx, my = (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2
        return (mx >= cx0) & (mx < cx1) & (my >= cy0) & (my < cy1)

    tin, bi, ai = inside(truth), inside(pb), inside(pa)
    fig, ax = plt.subplots(2, 2, figsize=(15, 11.5))
    draw(ax[0, 0], img, truth, pb, hb,
         f"pretrained DeepForest\n{int(hb.sum())} of {len(truth)} birds found",
         0.45, crop=(cx0, cy0, cx1, cy1))
    draw(ax[0, 1], img, truth, pa, ha,
         f"after training on the recovered annotations\n"
         f"{int(ha.sum())} of {len(truth)} birds found",
         0.45, crop=(cx0, cy0, cx1, cy1))
    draw(ax[1, 0], sub, truth[tin] - off, pb[bi] - off, hb[bi],
         f"close-up, {int(hb[bi].sum())} of {int(tin.sum())} found", 1.5)
    draw(ax[1, 1], sub, truth[tin] - off, pa[ai] - off, ha[ai],
         f"close-up, {int(ha[ai].sum())} of {int(tin.sum())} found", 1.5)
    fig.legend(handles=LEG, loc="lower center", ncol=3, fontsize=10, frameon=False)
    fig.suptitle(f"{frame}   {w}x{h}   held out, neither model trained on it. "
                 f"Both at score >= {SHOW_THRESH}.", fontsize=11)
    fig.tight_layout(rect=(0, 0.035, 1, 0.97))
    fig.savefig(f"{FIGS}/e4_frame_{r}_{frame}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  {r}. {frame}   truth {len(truth):4d}   "
          f"pre {int(hb.sum()):4d}/{len(pb):4d}   ft {int(ha.sum()):4d}/{len(pa):4d}",
          flush=True)

print(f"\nfigures in {FIGS}")

## 9. Reading the result

**Quote mAP@50 first.** It integrates over the whole confidence curve rather than fixing a
cutoff, so no threshold can flatter it, and it accumulates over the epoch, so the batch
size difference described in section 8 cannot reach it. It rose 2.4 times.

**Precision and recall rising together is the claim.** A model that only gained recall
would be drawing more boxes and catching a few more birds by volume. On the clearest
frame the fine-tuned model drew fewer boxes than the pretrained one, 828 against 1,103,
and found nearly twice as many birds.

**The absolute numbers are small and a reviewer will say so first.** An mAP@50 of 0.087 is
a weak detector and `map_small` is 0.011. Birds measure 16 to 54 pixels on these
photographs, which puts nearly all of them in the small-object regime where mAP is
punishing. The gain is real and the base is low; both belong in any sentence quoting it.

**What this does not say.** Every number here is scored against the recovered annotations,
so it measures agreement with the pipeline on unseen photographs, not accuracy against
people. A separate run scored a smaller version of this against 1,647 hand-placed dots and
fine-tuning lost there. Both results stand and they answer different questions.
`docs/training_analysis.md` carries the whole account.